In [178]:
import gurobipy as gp
#from gurobipy import Model, GRB
import re #Regular Expressions
import ast #Abstract syntax trees

In [179]:
#Criar o modelo
modelo = gp.Model("Agrupado")

*Varíaveis:*  
int K = ...  Número de níveis  
int J = ... Número de itens  
int B = ... Número de blocos de itens  
int H = ... Altura da gondôla  
int W = ... Largura da gondôla  
int S[Itens] = ... Conjunto de blocos de itens do tipo j  
int Wb[Blocos] = ... Largura do bloco b  
int Hb[Blocos] = ... Altura do bloco b em níveis  
int h[Itens] = ... Altura do item j  
int alfa[Blocos][Niveis][Niveis] = ...  

*Range:*  
Niveis = 1,...,K;  
Itens = 1,...,J;  
Blocos = 1,...,B;  


In [181]:
mu = 0.01 #Penalidade para minimizar a quantidade de z ativos

In [182]:
def read_opl_dat(filename):
    data = {}
    with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()

    # Remove comentários
    text = re.sub(r'/\*.*?\*/', '', text, flags=re.S) # Encontra padrão e troca 
    text = re.sub(r'//.*', '', text)

    # Divide por ";"
    entries = [e.strip() for e in text.split(';') if '=' in e]

    for entry in entries:
        name, value = entry.split('=', 1)
        name = name.strip()
        value = value.strip()

        # Substitui sintaxe de colchetes OPL por Python
        value = value.replace('{', '[').replace('}', ']')

        # Tenta converter para objeto Python
        try:
            data[name] = ast.literal_eval(value)
        except Exception:
            data[name] = value

    return data

# ler o .dat
dat = read_opl_dat("ModeloMercadoAgrupado2.dat")

# teste imprimindo algumas chaves
for k, v in dat.items():
    print(f"{k} = {type(v)} → {v if len(str(v)) < 120 else str(v)[:120]+'...'}")


K = <class 'int'> → 7
J = <class 'int'> → 4
B = <class 'int'> → 47
H = <class 'int'> → 1850
W = <class 'int'> → 560
S = <class 'list'> → [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22], [23, 24, 25, 26, 27, 28, 29, 30, 31,...
Wb = <class 'list'> → [240, 320, 400, 480, 160, 240, 320, 160, 240, 160, 160, 240, 320, 400, 480, 160, 240, 320, 160, 240, 160, 160, 240, 320,...
Hb = <class 'list'> → [2, 2, 2, 2, 3, 3, 3, 4, 4, 5, 6, 2, 2, 2, 2, 3, 3, 3, 4, 4, 5, 6, 2, 2, 2, 2, 3, 3, 3, 4, 4, 5, 6, 2, 2, 2, 2, 3, 3, 3,...
h = <class 'list'> → [260, 260, 260, 260]
w = <class 'list'> → [80, 80, 80, 80]
beta = <class 'list'> → [[66, 66, 99, 132, 132, 105, 0], [88, 88, 132, 176, 176, 140, 0], [110, 110, 165, 220, 220, 175, 0], [132, 132, 198, 264...
alfa = <class 'list'> → [[[1, 1, 0, 0, 0, 0, 0], [0, 1, 1, 0, 0, 0, 0], [0, 0, 1, 1, 0, 0, 0], [0, 0, 0, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 0], [0,...


In [183]:
K = dat['K']
J = dat['J']
B = dat['B']
H = dat['H']
W = dat['W']
S = dat['S']
Wb = dat['Wb']
Hb = dat['Hb']
h  = dat['h']
beta = dat['beta']
alfa = dat['alfa']
w = dat['w']

$$\text{Maximizar } Z = \sum_{K \in Niveis} \sum_{B \in Blocos} \beta_{b,k} \cdot x_{b,k} - \mu\sum_{B \in Blocos} \sum_{bl \in Blocos} z_{b,bl} $$

*Variáveis de Decisão:*  
float+ P[Blocos];  //Posição do bloco b na prateleira  
boolean x[Blocos][Niveis];  //1 se o bloco b for alocado no nível k; 0 Caso contrário.  
boolean z[Blocos][Blocos];  //1 se o bloco b é sequenciado antes do bloco b'; 0 caso contrário.  
float+ y[Niveis];  //Altura do nível k

In [186]:
niveis = range(K)
itens = range(J)
blocos = range(B)

P = modelo.addVars(blocos, name = "P")
x = modelo.addVars(blocos,niveis,vtype=gp.GRB.BINARY, name="x")
z = modelo.addVars(blocos,blocos,vtype=gp.GRB.BINARY, name="z")
y = modelo.addVars(niveis, name = "y")
print(niveis, itens, blocos)

range(0, 7) range(0, 4) range(0, 47)


In [187]:
modelo.setObjective(sum(beta[b][k]*x[b,k] for k in niveis for b in blocos) - 
                    mu*sum(z[b,bl] for b in blocos for bl in blocos), gp.GRB.MAXIMIZE)

*Restrições:*  
b-1 em todos contendo S[], o qual os dados inciaim em 1  
1. $\forall j\in\text{Itens}\ \sum_{b\in S_j}\sum_{k\in\text{Niveis}} x_{b,k}=1$

2. $\forall (j \in \text{Itens},\ b \in S[j],\ k, kl \in \text{Niveis} : kl \ge k \land \alpha_{b,k,kl} = 1)\quad h_j\,x_{b,k} \le y_{kl}$ 

3. $\sum_{k \in \text{Niveis}} y_k \le H$

4 e 5. $\forall (i, j \in \text{Itens},\ b \in S[i],\ bl \in S[j] : i \ne j)\ \{\, P_b + Wb_b \le P_{bl} + W(1 - z_{b,bl});\quad z_{b,bl} + z_{bl,b} \le 1 \,\}$ 
A largura da gondôla (W) foi utilizada como bigM

6. $\forall (b \in \text{Blocos},\ k \in \text{Niveis} : \alpha_{b,k,k} = -1)\quad x_{b,k} = 0$


In [189]:
modelo.addConstrs((gp.quicksum(x[b-1,k] for b in S[j] for k in niveis) == 1 for j in itens), name = "rest1")

for j in itens:
    for b in S[j]:
        for k in niveis:
            for kl in range(k,K):
                if alfa[b-1][k][kl] == 1:
                    modelo.addConstr((h[j]*x[b-1,k] <= y[kl]), name = "rest2")

modelo.addConstr((gp.quicksum(y[k] for k in niveis) <= H), name = "rest3")

for i in itens:
    for j in itens:
        if i != j:
            for b in S[i]:
                for bl in S[j]:
                    modelo.addConstr((P[b-1]+Wb[b-1] <= P[bl-1] + W*(1-z[b-1,bl-1])), name = "rest4")
                    modelo.addConstr((z[b-1,bl-1] + z[bl-1,b-1] <= 1), name = "rest5")

for b in blocos:
    for k in niveis:
        if alfa[b][k][k] == -1:
            modelo.addConstr(( x[b,k] == 0), name = "rest6")

*Restrições referentes ao cruzamento:*   

7. $\forall (i, j \in \text{Itens},\ b \in S[i],\ bl \in S[j],\ k \in \text{Niveis} : i \ne j \land (\alpha_{b,k,k} \ne -1 \lor \alpha_{bl,k,k} \ne -1))$  
$\sum_{\max(kl = k - Hb_b + 1| 0)}^{k} x_{b,kl} + \sum_{\max(kl = k - Hb_{bl} + 1| k+1)}^{k} x_{bl,kl} \le z_{b,bl} + z_{bl,b} + 1 $


8. $\forall (i \in \text{Itens},\ b \in S[i])\quad P_b + Wb_b \le W;$  

9. $\forall (b \in \text{Blocos},\ k \in \text{Niveis} : \alpha_{b,k,k} = -1)\quad x_{b,k} = 0;$  

10. $\forall (i \in \text{Itens},\ b, bl \in S[i])\quad z_{b,bl} = 0.$



In [191]:
for i in itens:
    for j in itens:
        if i!= j:
            for b in S[i]:
                for bl in S[j]:
                    for k in niveis:
                        if alfa[b-1][k][k] != -1 or alfa[bl-1][k][k] != -1:
                            modelo.addConstr((gp.quicksum(x[b-1,kl] for kl in range(max(k - Hb[b-1] + 1,0), k + 1)) + gp.quicksum(x[bl-1,kl] for kl in range(max(k - Hb[b-1] + 1,0), k + 1)) <= z[b-1,bl-1] + z[bl-1,b-1] + 1), name = "rest7")

for i in itens:
    for b in S[i]:
        modelo.addConstr((P[b-1] + Wb[b-1] <= W), name = "rest8")

for b in blocos:
    for k in niveis:
        if alfa[b][k][k] == -1:
            modelo.addConstr((x[b,k] == 0), name = "rest9")

for i in itens:
    for b in S[i]:
        for bl in S[i]:
            modelo.addConstr((z[b-1,bl-1] == 0), name = "rest10")

In [192]:
modelo.setParam("TimeLimit",60)
modelo.optimize()

for b in blocos: 
    for k in niveis:
        if x[b,k].X>0.5:
            print(f"bloco {b} ,niveis {k}, altura {Hb[b]}, largura {Wb[b]}, posicao incial {P[b].X}")

print("blocos adjacentes")
for b in blocos:
    for bl in blocos:
        if z[b,bl].X>0.5:
            print(b,bl)

Set parameter TimeLimit to value 60
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-10300H CPU @ 2.50GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  60

Academic license 2702680 - for non-commercial use only - registered to jo___@usp.br
Optimize a model with 13615 rows, 2592 columns and 70170 nonzeros
Model fingerprint: 0x835d8bae
Variable types: 54 continuous, 2538 integer (2538 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+02]
  Objective range  [1e-02, 3e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+03]
Presolve removed 10610 rows and 1247 columns
Presolve time: 0.27s
Presolved: 3005 rows, 1345 columns, 20706 nonzeros
Variable types: 49 continuous, 1296 integer (1296 binary)
Found heuristic solution: objective 383.9200000

Root relaxation: objective 1.026940e+03, 469 iterations, 

In [388]:
#fazer um dicionario percorrem o S pra saber o tipo de item
#corrigir o w minusculo, saida do modelo, escrever, perguntar pro elias


In [386]:
dados = dict()
lista_dados = list()


for i in itens:
    for j in range(len(S[i])):
        dados['bloco'] = S[i][j]
        dados['tipo'] = i+1
        lista_dados.append(dados.copy())


def search(n):
    for p in lista_dados:
        if p['bloco'] == n:
            return p

for b in blocos: 
    for k in niveis:
        if x[b,k].X>0.5:
            print(search(b))

{'bloco': 8, 'tipo': 1}
{'bloco': 16, 'tipo': 2}
{'bloco': 28, 'tipo': 3}
{'bloco': 42, 'tipo': 4}
